In [1]:
"""
E-Commerce Shoppers Intent Pipeline
Covers: Problem Statement & Data Description, EDA, Pre-processing, and Data Splitting.
Follows PEP 8 formatting standards.
"""

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Set publication-quality styling for EDA visualizations
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"font.size": 11, "figure.autolayout": True})


# ==============================================================================
# 1. DATA LOADING & DESCRIPTION
# ==============================================================================

# Load dataset (adjust path as needed)
df = pd.read_csv("~/Downloads/online_shoppers_intention.csv")

print("=" * 70)
print("1. DATASET OVERVIEW & DESCRIPTION")
print("=" * 70)
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print("\nData Types and Non-Null Counts:")
print(df.info())

print("\nTarget Class Distribution ('Revenue'):")
target_counts = df["Revenue"].value_counts()
target_pct = df["Revenue"].value_counts(normalize=True) * 100
for val in target_counts.index:
    print(f" Class {val}: {target_counts[val]} sessions ({target_pct[val]:.2f}%)")


# ==============================================================================
# 2. EXPLORATORY DATA ANALYSIS (EDA)
# ==============================================================================

print("\n" + "=" * 70)
print("2. EXPLORATORY DATA ANALYSIS")
print("=" * 70)

# Summary statistics for numerical metrics
print("\nNumerical Feature Summary Statistics:")
print(df.describe().T[["mean", "std", "min", "50%", "max"]])

# --- Chart 1: Conversion Rate by Visitor Type ---
fig1, ax1 = plt.subplots(figsize=(8, 5))
sns.barplot(
    data=df,
    x="VisitorType",
    y="Revenue",
    hue="VisitorType",
    legend=False,
    errorbar=None,
    palette="viridis",
    ax=ax1,
)
ax1.set_title(
    "Conversion Rate (Revenue = True) by Visitor Type",
    fontsize=14,
    fontweight="bold",
)
ax1.set_xlabel("Visitor Type", fontsize=12)
ax1.set_ylabel("Conversion Rate", fontsize=12)
ax1.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))

for p in ax1.patches:
    height = p.get_height()
    if height > 0:
        ax1.annotate(
            f"{height * 100:.1f}%",
            (p.get_x() + p.get_width() / 2.0, height),
            ha="center",
            va="center",
            xytext=(0, 8),
            textcoords="offset points",
            fontweight="bold",
        )
plt.show()

# --- Chart 2: Intent Signal (PageValues vs. BounceRates) ---
fig2, ax2 = plt.subplots(figsize=(9, 6))
sns.scatterplot(
    data=df,
    x="PageValues",
    y="BounceRates",
    hue="Revenue",
    style="Revenue",
    alpha=0.7,
    s=70,
    palette={False: "#d95f02", True: "#1b9e77"},
    ax=ax2,
)
ax2.set_title(
    "Session Intent: PageValues vs. Bounce Rates by Conversion",
    fontsize=14,
    fontweight="bold",
)
ax2.set_xlabel("Page Values", fontsize=12)
ax2.set_ylabel("Bounce Rates", fontsize=12)
plt.legend(title="Converted", labels=["False", "True"])
plt.show()


# ==============================================================================
# 3. DATA PRE-PROCESSING
# ==============================================================================

print("\n" + "=" * 70)
print("3. DATA PRE-PROCESSING")
print("=" * 70)

# A. Missing Values Handling
missing_vals = df.isnull().sum().sum()
print(f"Total missing values found: {missing_vals}")
if missing_vals > 0:
    df = df.dropna()  # Safety drop if missing values exist

# B. Outlier Treatment (99th percentile capping for high-variance durations)
duration_cols = [
    "Administrative_Duration",
    "Informational_Duration",
    "ProductRelated_Duration",
]
for col in duration_cols:
    cap_val = df[col].quantile(0.99)
    df[col] = np.where(df[col] > cap_val, cap_val, df[col])

# C. Feature Engineering & Correlated Features Interaction
df["Total_Pages"] = (
    df["Administrative"] + df["Informational"] + df["ProductRelated"]
)
df["Total_Duration"] = (
    df["Administrative_Duration"]
    + df["Informational_Duration"]
    + df["ProductRelated_Duration"]
)

# Derive average duration per page (handling division by zero)
df["Avg_Duration_Per_Page"] = np.where(
    df["Total_Pages"] > 0, df["Total_Duration"] / df["Total_Pages"], 0
)

# Address high correlation between BounceRates and ExitRates via interaction feature
df["Bounce_Exit_Interaction"] = df["BounceRates"] * df["ExitRates"]
df["Has_Page_Value"] = (df["PageValues"] > 0).astype(int)

# D. Categorical & Ordinal Mapping
df["Weekend"] = df["Weekend"].astype(int)
df["Revenue"] = df["Revenue"].astype(int)

month_map = {
    "Feb": 2,
    "Mar": 3,
    "May": 5,
    "June": 6,
    "Jul": 7,
    "Aug": 8,
    "Sep": 9,
    "Oct": 10,
    "Nov": 11,
    "Dec": 12,
}
df["Month_Numeric"] = df["Month"].map(month_map)

# Feature groupings
categorical_cols = ["VisitorType"]
numerical_cols = [
    "Administrative",
    "Administrative_Duration",
    "Informational",
    "Informational_Duration",
    "ProductRelated",
    "ProductRelated_Duration",
    "BounceRates",
    "ExitRates",
    "PageValues",
    "SpecialDay",
    "Month_Numeric",
    "OperatingSystems",
    "Browser",
    "Region",
    "TrafficType",
    "Total_Pages",
    "Total_Duration",
    "Avg_Duration_Per_Page",
    "Bounce_Exit_Interaction",
    "Has_Page_Value",
    "Weekend",
]

# E. Preprocessing Pipeline via ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(drop="first", sparse_output=False), categorical_cols),
    ]
)


# ==============================================================================
# 4. DATA SPLITTING (TRAINING, VALIDATION, AND TEST SETS)
# ==============================================================================

print("\n" + "=" * 70)
print("4. DATA SPLITTING")
print("=" * 70)

# Separate features (X) and target (y)
X = df.drop(columns=["Revenue", "Month"])
y = df["Revenue"]

# 1st Split: 85% Train/Val, 15% Test (Stratified by y)
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)

# 2nd Split: Split 85% Train/Val into 70% Train and 15% Validation overall
# (0.1765 * 0.85 ≈ 0.15 of total dataset)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.1765,
    random_state=42,
    stratify=y_train_val,
)

# Fit preprocessor strictly on training data to prevent data leakage
X_train_prep = preprocessor.fit_transform(X_train)
X_val_prep = preprocessor.transform(X_val)
X_test_prep = preprocessor.transform(X_test)

print(f"Total Dataset Count:      {len(df)}")
print(f"Training set shape:       {X_train_prep.shape} ({len(X_train)/len(df)*100:.1f}%)")
print(f"Validation set shape:     {X_val_prep.shape} ({len(X_val)/len(df)*100:.1f}%)")
print(f"Test set shape:           {X_test_prep.shape} ({len(X_test)/len(df)*100:.1f}%)")
print("\nStratification Check (Positive Target Rate across splits):")
print(f" Train positive rate:      {y_train.mean()*100:.2f}%")
print(f" Validation positive rate: {y_val.mean()*100:.2f}%")
print(f" Test positive rate:       {y_test.mean()*100:.2f}%")

FileNotFoundError: [Errno 2] No such file or directory: '/Users/samermarroki/Downloads/online_shoppers_intention.csv'